# VIGENT 파인튜닝 (가) — rf-detr 파이프라인 검증

**목적**: Roboflow 공개 데이터(office-dataset)로 rf-detr 파인튜닝이 **실제로 도는지** 끝까지 증명.
**환경**: Google Colab + 무료 GPU(T4).  **모델**: rf-detr Nano (Apache-2.0).

### 시작 전 — 꼭 하세요
1. 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU(T4)** 선택
2. 셀을 **위에서부터 순서대로** 실행(▶ 또는 Shift+Enter)
3. Roboflow API 키가 필요합니다(입력칸에 붙여넣기 — 노트북에 저장 안 됨)


## 1) 설치 (rf-detr · roboflow · supervision — 전부 permissive)


In [ ]:
!pip -q install rfdetr roboflow supervision


## 2) GPU 확인 (T4 가 잡혀야 정상)


In [ ]:
import torch
print('CUDA(GPU) 사용가능:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 → 런타임 유형을 GPU로 바꾸세요')


## 3) 데이터 다운로드 (office-dataset v5, COCO 형식)
- Roboflow API 키를 입력칸에 붙여넣으세요(가려져 입력됨, 저장 안 됨).
- 키 위치: roboflow.com → Settings → API Keys → Private API Key


In [ ]:
from getpass import getpass
from roboflow import Roboflow
API_KEY = getpass('Roboflow API 키를 붙여넣고 Enter: ')
rf = Roboflow(api_key=API_KEY)
project = rf.workspace('occupancy-detection').project('office-dataset')
dataset = project.version(5).download('coco')   # rf-detr 는 COCO 형식
DATASET_DIR = dataset.location
print('데이터 위치:', DATASET_DIR)
import os
print('구성:', os.listdir(DATASET_DIR))


## 4) 파인튜닝 (rf-detr Nano)
- 검증용이라 **epochs=15**(빠르게). 실제로는 30~50 권장.
- GPU 메모리 부족(OOM)이면 batch_size 를 4 로 낮추세요.


In [ ]:
from rfdetr import RFDETRNano
model = RFDETRNano()
model.train(
    dataset_dir=DATASET_DIR,
    epochs=15,
    batch_size=8,
    grad_accum_steps=2,
    lr=1e-4,
    output_dir='/content/output',
)
print('학습 완료')


## 5) 학습된 모델 찾기 + 다운로드
- rf-detr 는 output 폴더에 체크포인트를 저장합니다. 가장 좋은 것을 내려받습니다.


In [ ]:
import glob, os
ckpts = glob.glob('/content/output/**/*.pth', recursive=True)
print('생성된 체크포인트:')
for c in ckpts: print('  ', c, round(os.path.getsize(c)/1e6,1),'MB')
# 'best' 우선, 없으면 마지막
best = next((c for c in ckpts if 'best' in c.lower()), sorted(ckpts)[-1] if ckpts else None)
print('\n선택된 모델:', best)


In [ ]:
# 모델 파일 다운로드(브라우저로 저장됨)
from google.colab import files
import shutil
if best:
    out = '/content/vigent_office_finetuned.pth'
    shutil.copy(best, out)
    files.download(out)   # 브라우저 다운로드 시작
    print('다운로드:', out)
else:
    print('체크포인트를 못 찾음 — 학습 셀이 끝났는지 확인')


## 6) 다음 (맥에서)
1. 다운받은 `vigent_office_finetuned.pth` 를 `~/Desktop/VIGENT/vigent-core/weights/` 로 복사
2. Claude 에게 "이 모델 적용하고 정확도 재측정해줘" 라고 하세요
3. `eval_accuracy.py` 로 **재학습 전(P60%) vs 후** 비교 → 개선 확인

> 이 노트북이 끝까지 돌면 = 파인튜닝 파이프라인 검증 완료.
> 그다음 office-dataset 자리에 **회원님 현장 데이터**를 넣으면 실제 정확도가 올라갑니다.
